# The Magic Window — reproducible harness

**A stablecoin basis between two Bitcoin oracles.** Polymarket settles its hourly BTC market on the Binance BTC/USDT candle; Kalshi settles on a US-dollar index (CF Benchmarks BRTI). The gap between them is the **USDT/USD stablecoin basis** — signed and forecastable before the hour resolves.

This notebook reproduces the paper's headline on **1,350 hourly markets (26 May – 23 Jul 2026)**, scoring every outcome on **Kalshi's real published settlement** — not a reconstructed price.

**What it reproduces:** the offset = the USDT basis; and the direction rule that empties the double-loss tail (4.6% → 0.0%).  
**What it does *not* claim:** profitability. The pair is efficiently priced at rest; the remaining edge is a latency race this repo does not measure.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
GREEN,BLUE,RED,GRAY,AMBER = '#2c8a3c','#1c4478','#c0392b','#8a8f98','#f29121'

d = pd.read_csv('data/magic_window_1350.csv', parse_dates=['slot'])
N = len(d)
print(f'{N} settled hourly BTC markets, {d.slot.min().date()} to {d.slot.max().date()}')
d.head()

## 1 · The oracle gap *is* the stablecoin basis

`BTC/USD = BTC/USDT × (USDT/USD)`, so a price quoted in USDT sits above the same price in dollars by
`offset ≈ open × (1 − USDT/USD)`. We predict the offset from the USDT rate **at entry** and compare it to the offset that actually materialises at settlement.

In [ ]:
d['offset_pred'] = d.binance_open * (1 - d.usdt_usd)     # from USDT at entry
corr = np.corrcoef(d.offset_pred, d.offset_real)[0,1]
print(f"USDT/USD: mean {d.usdt_usd.mean():.5f} | max {d.usdt_usd.max():.5f} | below 1 always: {(d.usdt_usd<1).all()}")
print(f"offset predicted: ${d.offset_pred.mean():+.0f} | realized: ${d.offset_real.mean():+.0f} (sd ${d.offset_real.std():.0f})")
print(f"corr(pred, real): {corr:.2f} | residual sd ${(d.offset_real-d.offset_pred).std():.0f}")

fig,ax=plt.subplots(figsize=(6,5.2))
ax.scatter(d.offset_pred,d.offset_real,s=9,color=BLUE,alpha=.5,edgecolors='none')
lim=[min(d.offset_pred.min(),d.offset_real.min()),max(d.offset_pred.max(),d.offset_real.max())]
ax.plot(lim,lim,'--',color=GRAY,lw=1)
ax.set_xlabel('offset predicted from USDT at entry ($)'); ax.set_ylabel('offset realized at settlement ($)')
ax.set_title(f'The oracle gap is the stablecoin basis (corr {corr:.2f})'); ax.grid(alpha=.15); plt.show()

## 2 · The two rules — both scored on Kalshi's real result

Since `USDT/USD < 1` in every hour of the sample, the **offset-aware** rule is always **Polymarket "Up" + Kalshi "No"**, with the strike the nearest rung **above** the open. We re-derive its outcome from primitives:

- Polymarket **Up** wins ⟺ `binance_close ≥ binance_open` (deterministic)
- Kalshi **No** wins ⟺ `kalshi_result == 'no'` (**real published settlement**)

The **naive** rule takes the nearest strike on *either* side and is carried in the dataset for comparison.

In [ ]:
poly_up   = (d.binance_close >= d.binance_open).values
kalshi_no = (d.kalshi_result.str.lower() == 'no').values

def label(a,b):
    return pd.Series(np.where(a&b,'both_win', np.where(~a&~b,'both_lose','split')))

offa = label(poly_up, kalshi_no)
assert (offa.values == d.offsetaware_outcome.values).mean() > .999   # matches dataset

def dist(s):
    v=s.value_counts(normalize=True).mul(100)
    return {k:round(v.get(k,0.),1) for k in ['both_win','split','both_lose']}

tbl = pd.DataFrame({'naive':dist(d.naive_outcome),'offset_aware':dist(offa)}).T
tbl = tbl[['both_win','split','both_lose']]
print(tbl.to_string())
print(f"\ndouble loss 4.6% -> 0.0%  (0 of {N}; rule-of-three 95% upper bound {3/N*100:.2f}%)")

## 3 · Why the tail vanishes

The naive rule's double losses are **not scattered** — every one is an hour where the nearest strike sat **below** the open. The offset-aware rule refuses a strike below the open, so those flips turn into double **wins**. That part of the zero is **arithmetic**, not luck.

In [ ]:
dl = d[d.naive_outcome=='both_lose']
print(f'naive double losses: {len(dl)}')
print(f'all with nearest strike below the open: {(dl.strike_naive < dl.binance_open).all()}')

## 4 · The double-loss corner is empty

Each hour by how its two legs resolved. A double loss needs the top-left corner (Polymarket Up loses **and** Kalshi No loses). No hour is there.

In [ ]:
x=(d.binance_close-d.binance_open).values
y=(d.brti_proxy-d.strike_offsetaware).values
win=offa.values=='both_win'
fig,ax=plt.subplots(figsize=(7,6))
lo_x,hi_x=x.min()-40,x.max()+40; lo_y,hi_y=y.min()-40,y.max()+40
ax.axhspan(0,hi_y,xmin=0,xmax=(0-lo_x)/(hi_x-lo_x),color=RED,alpha=.07)
ax.axvline(0,color='k',lw=.8); ax.axhline(0,color='k',lw=.8)
ax.scatter(x[~win],y[~win],s=8,color=BLUE,alpha=.5,edgecolors='none',label=f'split ({(~win).sum()})')
ax.scatter(x[win],y[win],s=8,color=GREEN,alpha=.6,edgecolors='none',label=f'double win ({win.sum()})')
ax.text(lo_x*.95,hi_y*.85,'DOUBLE-LOSS quadrant\nEMPTY (0 of 1,350)',color=RED,fontsize=9,fontweight='bold',va='top')
ax.set_xlim(lo_x,hi_x); ax.set_ylim(lo_y,hi_y)
ax.set_xlabel('Binance close - open  (>0: Polymarket Up wins)')
ax.set_ylabel('BRTI - strike  (>=0: Kalshi No loses)')
ax.set_title('The double-loss corner is empty'); ax.legend(loc='lower left',fontsize=8.5); ax.grid(alpha=.15); plt.show()

---
**Scope.** This reproduces the *validated structure* on real settlement: the offset is the USDT basis, and the direction rule removes the double-loss tail. It is **not** a profitability claim — at rest the pair is fairly priced (`Σ = up_ask + no_ask ≈ 1 + P(magic window) ≥ 1`); any edge is a sub-second latency race, which this repository does not measure.

Data: `data/magic_window_1350.csv` — 1,350 hourly BTC markets with Kalshi's real published result. See the paper for the full treatment.